# ABSA Training — T4 GPU / CPU (Google Colab)

Full pipeline: data prep → feature engineering → DeBERTa fine-tuning → inference demo.

**Recommended: T4 GPU** → Runtime → Change runtime type → T4 GPU (~8–15 min for 3 epochs)  
**CPU fallback** → set `FORCE_CPU = True` in Cell 3 (~80–120 min for 3 epochs)

Run cells **top to bottom** in order.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q transformers scikit-learn pandas pyyaml openpyxl tabulate accelerate

In [1]:
# Cell 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Cell 3 — Project path & config
import os, sys, json, time
from pathlib import Path

# ── Set FORCE_CPU = True only if you want CPU instead of T4 GPU ──────────
FORCE_CPU = False
if FORCE_CPU:
    os.environ["CUDA_VISIBLE_DEVICES"] = ""

PROJECT_DIR = Path("/content/drive/MyDrive/1_VNIT/Semester_3/NLP/NLP_Project"
                   "/CustomerSentimentAnalysis/CustomerSentimentAnalysis")

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"Project not found at {PROJECT_DIR}\n"
        "Check Drive is mounted and the path above matches your folder structure."
    )

os.chdir(str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR))

NUM_EPOCHS    = 10          # 10 epochs for best results; reduce to 3 for a quick test
BATCH_SIZE    = 16         # 16 on T4 GPU; reduce to 8 if CPU or low RAM
MAX_SEQ_LEN   = 64
LEARNING_RATE = 2e-5
PRETRAINED    = "yangheng/deberta-v3-base-absa-v1.1"

MODEL_DIR    = str(PROJECT_DIR / "models" / "asc_model")
RESULTS_FILE = str(PROJECT_DIR / "models" / "training_results.json")
TRAIN_FILE   = str(PROJECT_DIR / "data" / "processed" / "train_data.txt")
VAL_FILE     = str(PROJECT_DIR / "data" / "processed" / "val_data.txt")
TEST_FILE    = str(PROJECT_DIR / "data" / "processed" / "test_data.txt")

LABEL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}
ID2LABEL  = {0: 'negative', 1: 'neutral', 2: 'positive'}

def banner(title):
    print(f"\n{'='*60}\n  {title}\n{'='*60}")

import torch
device = "GPU ✓" if torch.cuda.is_available() else "CPU"
print(f"Device  : {device}")
print(f"Project : {PROJECT_DIR}")
print(f"Model   : {PRETRAINED}")
print(f"Epochs  : {NUM_EPOCHS}  |  Batch: {BATCH_SIZE}  |  MaxSeq: {MAX_SEQ_LEN}")

Device  : CPU
Project : /content/drive/MyDrive/1_VNIT/Semester_3/NLP/NLP_Project/CustomerSentimentAnalysis/CustomerSentimentAnalysis
Model   : yangheng/deberta-v3-base-absa-v1.1
Epochs  : 10  |  Batch: 16  |  MaxSeq: 64


In [3]:
# Cell 4 — Data Preparation
banner("Step 1 — Data Preparation")

from src.data.prepare import DataPreparator
DataPreparator(input_path="data/raw", output_path="data/processed").run()


  Step 1 — Data Preparation


In [4]:
# Cell 5 — Feature Engineering
banner("Step 2 — Feature Engineering")

from src.features.build_features import FeatureBuilder
FeatureBuilder(
    input_path="data/processed",
    output_path="data/processed",
    test_size=0.1,
    val_size=0.1,
).run()


  Step 2 — Feature Engineering


In [5]:
# Cell 6 — Verify data format & labels
banner("Step 3 — Data Verification")

os.makedirs(MODEL_DIR, exist_ok=True)

def count_and_check(path):
    count = 0
    labels = set()
    with open(path, encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('$$$')
            if len(parts) == 3:
                labels.add(parts[2].strip().lower())
                count += 1
    return count, labels

n_train, l_train = count_and_check(TRAIN_FILE)
n_val,   l_val   = count_and_check(VAL_FILE)
n_test,  l_test  = count_and_check(TEST_FILE)
all_labels = l_train | l_val | l_test

print(f"train={n_train}  val={n_val}  test={n_test}")
print(f"Labels : {sorted(all_labels)}")

unexpected = all_labels - {'positive', 'negative', 'neutral'}
if unexpected:
    print(f"WARNING: unexpected labels found: {unexpected}")
else:
    print("  Labels OK — ready for training")

with open(TRAIN_FILE, encoding='utf-8') as f:
    print(f"\nSample: {f.readline().strip()[:100]}")


  Step 3 — Data Verification
train=1416  val=177  test=177
Labels : ['negative', 'neutral', 'positive']
  Labels OK — ready for training

Sample: मुझे really दोनों scallops और mahi mahi (on saffron risotto -yum!) पसंद हैंं।$$$[scallops]$$$positiv


In [6]:
# Cell 7 — Load tokenizer & model
banner("Step 4a — Loading DeBERTa model (first run ~750 MB download)")

import torch
import numpy as np
from torch.utils.data import Dataset as TorchDataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import accuracy_score, f1_score

tokenizer = AutoTokenizer.from_pretrained(PRETRAINED)
model = AutoModelForSequenceClassification.from_pretrained(
    PRETRAINED,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)
print(f"Model loaded: {model.config.model_type}  |  Output classes: {model.config.num_labels}")


  Step 4a — Loading DeBERTa model (first run ~750 MB download)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/372 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/738M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: yangheng/deberta-v3-base-absa-v1.1
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded: deberta-v2  |  Output classes: 3


In [7]:
# Cell 8 — Build datasets
banner("Step 4b — Tokenizing data")

def read_triplets(path):
    """Read text$$$[aspect]$$$sentiment — rsplit from right handles $$$ in text."""
    texts, aspects, labels = [], [], []
    skipped = 0
    with open(path, encoding='utf-8') as f:
        for line in f:
            parts = line.strip().rsplit('$$$', maxsplit=2)   # split from RIGHT
            if len(parts) == 3:
                text, aspect, polarity = parts
                aspect  = aspect.strip().strip('[]')
                polarity = polarity.strip().lower()
                if polarity in LABEL2ID:
                    texts.append(text.strip())
                    aspects.append(aspect)
                    labels.append(LABEL2ID[polarity])
                else:
                    skipped += 1
            else:
                skipped += 1
    if skipped:
        print(f"  Skipped {skipped} malformed lines in {path}")
    return texts, aspects, labels

class ABSADataset(TorchDataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

def encode(texts, aspects):
    return tokenizer(
        texts, aspects,
        truncation=True, padding='max_length', max_length=MAX_SEQ_LEN
    )

train_texts, train_aspects, train_labels = read_triplets(TRAIN_FILE)
val_texts,   val_aspects,   val_labels   = read_triplets(VAL_FILE)
test_texts,  test_aspects,  test_labels  = read_triplets(TEST_FILE)

print(f"Samples — train: {len(train_labels)}  val: {len(val_labels)}  test: {len(test_labels)}")

# Show class distribution
from collections import Counter
dist = Counter(train_labels)
print(f"Train distribution — negative: {dist[0]}  neutral: {dist[1]}  positive: {dist[2]}")


# Balance training set — oversample negative & neutral up to positive count
import random
random.seed(42)
dist = Counter(train_labels)
max_count = max(dist.values())
balanced_texts, balanced_aspects, balanced_labels = [], [], []
for cls in [0, 1, 2]:
    idx = [i for i, l in enumerate(train_labels) if l == cls]
    for i in random.choices(idx, k=max_count):
        balanced_texts.append(train_texts[i])
        balanced_aspects.append(train_aspects[i])
        balanced_labels.append(train_labels[i])
train_texts, train_aspects, train_labels = balanced_texts, balanced_aspects, balanced_labels
print(f'After oversampling: {len(train_labels)} samples (each class = {max_count})')
print("Tokenizing...")
train_dataset = ABSADataset(encode(train_texts, train_aspects), train_labels)
val_dataset   = ABSADataset(encode(val_texts,   val_aspects),   val_labels)
test_dataset  = ABSADataset(encode(test_texts,  test_aspects),  test_labels)
print("Done")


  Step 4b — Tokenizing data
Samples — train: 1416  val: 177  test: 177
Train distribution — negative: 267  neutral: 249  positive: 900
After oversampling: 2700 samples (each class = 900)
Tokenizing...
Done


In [ ]:
# Cell 9 — Train
banner("Step 4c — Training")

from sklearn.utils.class_weight import compute_class_weight

# Class weights — fixes positive-class dominance (~56% of data)
_cw = compute_class_weight('balanced', classes=np.array([0, 1, 2]), y=np.array(train_labels))
class_weights = torch.tensor(_cw, dtype=torch.float)
print(f"Class weights: negative={_cw[0]:.2f}  neutral={_cw[1]:.2f}  positive={_cw[2]:.2f}")

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(
            outputs.logits, labels,
            weight=class_weights.to(outputs.logits.device)
        )
        return (loss, outputs) if return_outputs else loss

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    return {
        'accuracy': accuracy_score(p.label_ids, preds),
        'f1_macro': f1_score(p.label_ids, preds, average='macro'),
    }

training_args = TrainingArguments(
    output_dir=MODEL_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    seed=42,
    report_to='none',
    logging_steps=20,
)

hf_trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

t0 = time.time()
hf_trainer.train()
elapsed = time.time() - t0
print(f"\nTraining complete in {elapsed/60:.1f} min")

test_results = hf_trainer.evaluate(test_dataset)
print(f"\nTest results: {test_results}")

hf_trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

os.makedirs(os.path.dirname(RESULTS_FILE), exist_ok=True)
with open(RESULTS_FILE, 'w', encoding='utf-8') as mf:
    json.dump({'metrics': test_results, 'elapsed_min': elapsed / 60}, mf, indent=2)

print(f"Model   saved: {MODEL_DIR}")
print(f"Metrics saved: {RESULTS_FILE}")


  Step 4c — Training
Class weights: negative=1.00  neutral=1.00  positive=1.00


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.015385,0.932942,0.587571,0.490039


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
# Cell 10 — Inference Demo
banner("Step 5 — Inference Demo")

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

inf_tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
inf_model     = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
inf_model.eval()

def predict_sentiment(text, aspect):
    inputs = inf_tokenizer(
        text, aspect,
        return_tensors='pt', truncation=True, max_length=MAX_SEQ_LEN
    )
    with torch.no_grad():
        logits = inf_model(**inputs).logits
    return ID2LABEL[logits.argmax(-1).item()]

test_cases = [
    ("Great food but the service was very slow.",    "food"),
    ("Great food but the service was very slow.",    "service"),
    ("Staff was friendly and ambiance was amazing!", "ambiance"),
    ("Too expensive for such cold food.",            "price"),
    ("Excellent pasta and wine selection.",          "pasta"),
    ("\u0916\u093e\u0928\u093e \u092c\u0939\u0941\u0924 \u0938\u094d\u0935\u093e\u0926\u093f\u0937\u094d\u091f \u0925\u093e \u0932\u0947\u0915\u093f\u0928 \u0938\u0947\u0935\u093e \u0916\u0930\u093e\u092c \u0925\u0940\u0964", "\u0916\u093e\u0928\u093e"),
]

print(f"{'Aspect':<15}  {'Sentiment':<12}  Review")
print("-" * 70)
for text, aspect in test_cases:
    sentiment = predict_sentiment(text, aspect)
    print(f"  {aspect:<13}  {sentiment:<12}  {text[:55]}")

banner("Done")
print(f"  Model  : {MODEL_DIR}")
print(f"  Results: {RESULTS_FILE}")


  Step 5 — Inference Demo


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Aspect           Sentiment     Review
----------------------------------------------------------------------
  food           positive      Great food but the service was very slow.
  service        positive      Great food but the service was very slow.
  ambiance       positive      Staff was friendly and ambiance was amazing!
  price          positive      Too expensive for such cold food.
  pasta          positive      Excellent pasta and wine selection.
  खाना           negative      खाना बहुत स्वादिष्ट था लेकिन सेवा खराब थी।

  Done
  Model  : /content/drive/MyDrive/1_VNIT/Semester_3/NLP/NLP_Project/CustomerSentimentAnalysis/CustomerSentimentAnalysis/models/asc_model
  Results: /content/drive/MyDrive/1_VNIT/Semester_3/NLP/NLP_Project/CustomerSentimentAnalysis/CustomerSentimentAnalysis/models/training_results.json


In [ ]:
# Cell 11 — Generate presentation charts & metrics
banner("Step 6 — Generating Results for Presentation")

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, f1_score, precision_score, recall_score
)

RESULTS_OUT = PROJECT_DIR / "results_v1"
RESULTS_OUT.mkdir(exist_ok=True)
CLASSES = ['negative', 'neutral', 'positive']

# ── Load model & run predictions ────────────────────────────────────────────
inf_tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
inf_model     = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
inf_model.eval()
if torch.cuda.is_available():
    inf_model = inf_model.cuda()

def read_triplets_safe(path):
    texts, aspects, labels = [], [], []
    total = 0
    with open(path, encoding='utf-8') as f:
        for line in f:
            total += 1
            parts = line.strip().rsplit('$$$', maxsplit=2)  # split from RIGHT
            if len(parts) == 3:
                text, aspect, polarity = parts
                polarity = polarity.strip().lower()
                if polarity in LABEL2ID:
                    texts.append(text.strip())
                    aspects.append(aspect.strip().strip('[]'))
                    labels.append(LABEL2ID[polarity])
    print(f"  {path.split('/')[-1]}: {total} lines → {len(labels)} valid samples")
    return texts, aspects, labels

test_texts, test_aspects, true_labels = read_triplets_safe(TEST_FILE)
print(f"\nTest set: {len(true_labels)} samples")

from collections import Counter
dist = Counter(true_labels)
print(f"Class distribution — negative: {dist[0]}  neutral: {dist[1]}  positive: {dist[2]}")

pred_labels = []
for text, aspect in zip(test_texts, test_aspects):
    inputs = inf_tokenizer(
        text, aspect, return_tensors='pt',
        truncation=True, max_length=MAX_SEQ_LEN
    )
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    with torch.no_grad():
        logits = inf_model(**inputs).logits
    pred_labels.append(logits.argmax(-1).item())

true_arr = np.array(true_labels)
pred_arr = np.array(pred_labels)

pred_dist = Counter(pred_labels)
print(f"Predictions    — negative: {pred_dist[0]}  neutral: {pred_dist[1]}  positive: {pred_dist[2]}")

accuracy = accuracy_score(true_arr, pred_arr)
f1_macro = f1_score(true_arr, pred_arr, average='macro', zero_division=0)
precision = precision_score(true_arr, pred_arr, average=None, labels=[0,1,2], zero_division=0)
recall    = recall_score(true_arr,    pred_arr, average=None, labels=[0,1,2], zero_division=0)
f1_class  = f1_score(true_arr,        pred_arr, average=None, labels=[0,1,2], zero_division=0)
report    = classification_report(true_arr, pred_arr, target_names=CLASSES, zero_division=0)

print(f"\nAccuracy: {accuracy:.4f}  |  F1 Macro: {f1_macro:.4f}")
print(report)

# ── 1. Confusion Matrix ──────────────────────────────────────────────────────
cm     = confusion_matrix(true_arr, pred_arr, labels=[0,1,2])
cm_pct = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9) * 100

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=False, cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES,
            ax=ax, linewidths=0.5, linecolor='white')
for i in range(3):
    for j in range(3):
        dark = cm[i,j] > cm.max() * 0.5
        ax.text(j+0.5, i+0.35, str(cm[i,j]), ha='center', va='center',
                fontsize=15, fontweight='bold', color='white' if dark else 'black')
        ax.text(j+0.5, i+0.65, f"({cm_pct[i,j]:.1f}%)", ha='center', va='center',
                fontsize=10, color='white' if dark else 'grey')
ax.set_xlabel('Predicted Label', fontsize=13, labelpad=10)
ax.set_ylabel('True Label', fontsize=13, labelpad=10)
ax.set_title('Confusion Matrix — ABSA Sentiment Classification', fontsize=14, pad=15)
plt.tight_layout()
plt.savefig(str(RESULTS_OUT / 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show(); print("Saved: confusion_matrix.png")

# ── 2. Per-class Precision / Recall / F1 ────────────────────────────────────
x, w = np.arange(3), 0.25
fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x-w, precision, w, label='Precision', color='#3498db', alpha=0.85)
b2 = ax.bar(x,   recall,    w, label='Recall',    color='#e67e22', alpha=0.85)
b3 = ax.bar(x+w, f1_class,  w, label='F1-Score',  color='#2ecc71', alpha=0.85)
for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        if h > 0.01:
            ax.text(bar.get_x()+bar.get_width()/2, h+0.01, f'{h:.2f}',
                    ha='center', va='bottom', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels([c.capitalize() for c in CLASSES], fontsize=12)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Per-Class Metrics — Precision, Recall & F1', fontsize=14, pad=12)
ax.legend(fontsize=11)
ax.axhline(0.5, color='grey', linestyle='--', linewidth=0.8, alpha=0.5)
ax.grid(axis='y', alpha=0.3)

# annotate support counts on x-axis
for i, cls in enumerate(CLASSES):
    ax.text(i, -0.12, f"n={dist[i]}", ha='center', fontsize=9,
            color='grey', transform=ax.get_xaxis_transform())
plt.tight_layout()
plt.savefig(str(RESULTS_OUT / 'per_class_metrics.png'), dpi=150, bbox_inches='tight')
plt.show(); print("Saved: per_class_metrics.png")

# ── 3. Overall Metrics Summary Card ─────────────────────────────────────────
elapsed_min = None
if Path(RESULTS_FILE).exists():
    with open(RESULTS_FILE) as f:
        elapsed_min = json.load(f).get('elapsed_min')

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
for ax, (title, val, color, label) in zip(axes, [
    ('Accuracy',   accuracy,              '#3498db', f'{accuracy*100:.1f}%'),
    ('F1 Macro',   f1_macro,              '#9b59b6', f'{f1_macro:.3f}'),
    ('Test Samples', len(true_arr)/max(len(true_arr),1), '#1abc9c', f'{len(true_arr)} samples'),
]):
    ax.pie([val, max(0, 1-val)], colors=[color, '#ecf0f1'],
           startangle=90, wedgeprops=dict(width=0.45))
    ax.text(0, 0, label, ha='center', va='center',
            fontsize=14, fontweight='bold', color=color)
    ax.set_title(title, fontsize=12, pad=8)
time_str = f"  |  Train time: {elapsed_min:.1f} min" if elapsed_min else ""
fig.suptitle(f'Model Performance — DeBERTa-v3-base ABSA{time_str}', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(str(RESULTS_OUT / 'overall_metrics.png'), dpi=150, bbox_inches='tight')
plt.show(); print("Saved: overall_metrics.png")

# ── 4. Save text reports ─────────────────────────────────────────────────────
with open(RESULTS_OUT / 'classification_report.txt', 'w', encoding='utf-8') as f:
    f.write("ABSA Sentiment Classification — Test Set Results\n")
    f.write("=" * 52 + "\n")
    f.write(f"Model     : yangheng/deberta-v3-base-absa-v1.1\n")
    f.write(f"Test size : {len(true_arr)} samples\n")
    if elapsed_min:
        f.write(f"Train time: {elapsed_min:.1f} min\n")
    f.write("=" * 52 + "\n\n")
    f.write(report)
    f.write(f"\nAccuracy  : {accuracy:.4f}\nF1 Macro  : {f1_macro:.4f}\n")

with open(RESULTS_OUT / 'predictions_sample.txt', 'w', encoding='utf-8') as f:
    f.write(f"{'Text':<55}  {'Aspect':<12}  {'True':<10}  {'Pred':<10}  Match\n")
    f.write("-" * 105 + "\n")
    for i in range(min(20, len(test_texts))):
        t, p = ID2LABEL[true_labels[i]], ID2LABEL[pred_labels[i]]
        f.write(f"{test_texts[i][:54]:<55}  {test_aspects[i]:<12}  {t:<10}  {p:<10}  {'✓' if t==p else '✗'}\n")

print(f"""
All results saved to: results/
  ├── confusion_matrix.png
  ├── per_class_metrics.png
  ├── overall_metrics.png
  ├── classification_report.txt
  └── predictions_sample.txt
""")